In [7]:
import pandas as pd
import sqlite3
import os

def load_to_sqlite(csv_folder=r"D:/VSCode/LLM NL-SQL/Data", db_path="data/olist.db"):
    # Ensure the database folder exists
    os.makedirs(os.path.dirname(db_path), exist_ok=True)

    conn = sqlite3.connect(db_path)

    for file in os.listdir(csv_folder):
        if file.lower().endswith(".csv"):
            table_name = file.replace(".csv", "")
            df = pd.read_csv(os.path.join(csv_folder, file))
            df.to_sql(table_name, conn, if_exists="replace", index=False)
            print(f"✅ Loaded {file} into table {table_name}")

    # List all tables to verify
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    print("📋 Tables in database:", tables)

    # Show row counts for each table
    for (t,) in tables:
        cursor.execute(f"SELECT COUNT(*) FROM {t};")
        count = cursor.fetchone()[0]
        print(f"   {t}: {count} rows")

    conn.close()
    print("🎉 All CSVs loaded successfully.")


In [8]:
load_to_sqlite()


✅ Loaded olist_customers_dataset.csv into table olist_customers_dataset
✅ Loaded olist_geolocation_dataset.csv into table olist_geolocation_dataset
✅ Loaded olist_orders_dataset.csv into table olist_orders_dataset
✅ Loaded olist_order_items_dataset.csv into table olist_order_items_dataset
✅ Loaded olist_order_payments_dataset.csv into table olist_order_payments_dataset
✅ Loaded olist_order_reviews_dataset.csv into table olist_order_reviews_dataset
✅ Loaded olist_products_dataset.csv into table olist_products_dataset
✅ Loaded olist_sellers_dataset.csv into table olist_sellers_dataset
✅ Loaded product_category_name_translation.csv into table product_category_name_translation
📋 Tables in database: [('olist_customers_dataset',), ('olist_geolocation_dataset',), ('olist_orders_dataset',), ('olist_order_items_dataset',), ('olist_order_payments_dataset',), ('olist_order_reviews_dataset',), ('olist_products_dataset',), ('olist_sellers_dataset',), ('product_category_name_translation',)]
   olist

In [10]:
import sqlite3

conn = sqlite3.connect("data/olist.db")
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("Tables:", cursor.fetchall())
conn.close()


Tables: [('olist_customers_dataset',), ('olist_geolocation_dataset',), ('olist_orders_dataset',), ('olist_order_items_dataset',), ('olist_order_payments_dataset',), ('olist_order_reviews_dataset',), ('olist_products_dataset',), ('olist_sellers_dataset',), ('product_category_name_translation',)]


In [11]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("data/olist.db")
df = pd.read_sql("SELECT * FROM olist_orders_dataset LIMIT 5;", conn)
print(df)
conn.close()


                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08

In [14]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("data/olist.db")

query = """
SELECT t.product_category_name_english AS category,
       COUNT(*) AS items_sold
FROM olist_order_items_dataset oi
JOIN olist_products_dataset p
  ON oi.product_id = p.product_id
JOIN product_category_name_translation t
  ON p.product_category_name = t.product_category_name
GROUP BY category
ORDER BY items_sold DESC
LIMIT 5;
"""

df = pd.read_sql(query, conn)
print(df)

conn.close()


                category  items_sold
0         bed_bath_table       11115
1          health_beauty        9670
2         sports_leisure        8641
3        furniture_decor        8334
4  computers_accessories        7827


In [15]:
conn = sqlite3.connect("data/olist.db")
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM olist_order_reviews_dataset;")
print("Reviews count:", cursor.fetchone()[0])
conn.close()


Reviews count: 99224
